# NCDB collision dataset validation

This companion inspects the saved build outputs without downloading files or rerunning the full build. It checks collision keys, road-tractor selection, annual totals, and quality flags, then compares counts with published Transport Canada benchmarks.

**Scope:** police-reported fatal and injury collisions on Canadian public roads. Road tractor means NCDB `V_TYPE=08`, with or without a semitrailer. A collision involving several tractors still counts once. This is a validation notebook, not a statistical trend analysis.

The build status and unresolved differences below control how the dataset may be used. Passing structural checks does not establish agreement with an external publication.

## Setup and source outputs

Run with this repository's Python environment (`pip install -e '.[notebook]'`). First run `python -m truck_collisions.build` from the repository root if the outputs are missing. Inputs are the original files documented in `docs/ncdb_acquisition.md`, the local archive inventory, and the source manifest saved by the build. The published benchmark is recorded in `data/reference/tractor_collision_benchmark.csv`.

Keys use year and case ID together. Source codes remain strings, including unknown and special codes. No casualty totals are calculated here: person placeholders and conflicting person records need a separate validation design.

In [1]:
from pathlib import Path
import json
from html import escape
import duckdb
from IPython.display import display, HTML, Markdown

root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pyproject.toml").exists() and (p / "src/truck_collisions").exists()), None)
assert root is not None, "Open this notebook from within the repository."
output = root / "data/derived/ncdb"
required = ["validation.json", "annual_counts.csv", "reconciliation.csv", "quality_by_year.csv",
            "collisions.parquet", "road_tractor_collisions.parquet", "road_tractor_collisions_strict.parquet", "special_vehicle_codes.csv"]
assert all((output / name).exists() for name in required), "Run the dataset build first."
validation = json.loads((output / "validation.json").read_text())
connection = duckdb.connect()

def literal(value):
    return "'" + str(value).replace("'", "''") + "'"

for view, name, reader in [("collisions", "collisions.parquet", "read_parquet"),
                           ("tractors", "road_tractor_collisions.parquet", "read_parquet"),
                           ("strict_tractors", "road_tractor_collisions_strict.parquet", "read_parquet"),
                           ("annual", "annual_counts.csv", "read_csv_auto"),
                           ("reconciliation", "reconciliation.csv", "read_csv_auto"),
                           ("quality", "quality_by_year.csv", "read_csv_auto"),
                           ("special", "special_vehicle_codes.csv", "read_csv_auto")]:
    connection.execute(f"CREATE VIEW {view} AS SELECT * FROM {reader}({literal(output / name)})")

def show_table(query):
    result = connection.execute(query)
    headings = [column[0] for column in result.description]
    rows = result.fetchall()
    header = "".join(f"<th>{escape(name.replace('_', ' '))}</th>" for name in headings)
    def cell(value, heading):
        return "—" if value is None else (f"{value:,}" if isinstance(value, int) and not isinstance(value, bool) and "year" not in heading else str(value))
    body = "".join("<tr>" + "".join(f"<td>{escape(cell(value, heading))}</td>" for heading, value in zip(headings, row)) + "</tr>" for row in rows)
    display(HTML(f'<div style="overflow-x:auto"><table style="border-collapse:collapse;font-size:13px"><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table></div>'))

display(Markdown(f"**Build status:** `{validation['status']}`  \n**Built (UTC):** {validation['created_at_utc']}  \n**Recorded DuckDB version:** {validation['duckdb_version']}"))

**Build status:** `partial_structural_exclusions_review_required`  
**Built (UTC):** 2026-10-15T06:01:31.657216+00:00  
**Recorded DuckDB version:** 1.5.6

In [2]:
for check, value in validation['checks'].items():
    print(f"{check}: {value}")

collision_key_unique: True
invalid_key_record_rows: 0
invalid_collision_events: 3
excluded_tractor_events: 11
external_mismatch_years: [2013, 2015, 2016, 2017, 2019]


## Independent structural checks

These assertions inspect the Parquet outputs and compare annual counts with the build summary. The primary tractor table preserves all observed tractor events and their flags. The strict sensitivity subset requires the build's candidate criteria. A successful cell means these particular checks passed; it does not resolve source conflicts or external count differences.

In [3]:
all_keys = connection.execute("SELECT count(*), count(DISTINCT (C_YEAR,C_CASE)), count(*) FILTER (WHERE C_YEAR IS NULL OR C_CASE IS NULL) FROM collisions").fetchone()
assert all_keys[0] == all_keys[1] and all_keys[2] == 0, "Collision key check failed."
assert connection.execute("SELECT count(*)=count(DISTINCT (C_YEAR,try_cast(C_CASE AS HUGEINT))) AND count(*) FILTER (WHERE try_cast(C_CASE AS HUGEINT) IS NULL)=0 FROM collisions").fetchone()[0], "Numeric-equivalent case IDs duplicate within a year."
for view, predicate, annual_column in [
    ("tractors", "is_road_tractor", "observed_tractor_collisions"),
    ("strict_tractors", "is_road_tractor AND is_analysis_candidate", "strict_tractor_collisions")]:
    keys = connection.execute(f"SELECT count(*),count(DISTINCT (C_YEAR,C_CASE)),count(*) FILTER (WHERE NOT is_road_tractor OR is_road_tractor IS NULL) FROM {view}").fetchone()
    assert keys[0] == keys[1] and keys[2] == 0, f"{view} key or tractor flag check failed."
    for left, right in [(f"SELECT C_YEAR,C_CASE FROM {view}", f"SELECT C_YEAR,C_CASE FROM collisions WHERE {predicate}"),
                        (f"SELECT C_YEAR,C_CASE FROM collisions WHERE {predicate}", f"SELECT C_YEAR,C_CASE FROM {view}")]:
        assert connection.execute(f"SELECT count(*) FROM ({left} EXCEPT {right})").fetchone()[0] == 0
    assert connection.execute(f"SELECT count(*) FROM (SELECT C_YEAR AS year,count(*) AS n FROM {view} GROUP BY C_YEAR) t FULL JOIN annual a USING(year) WHERE coalesce(t.n,0) <> coalesce(a.{annual_column},0)").fetchone()[0] == 0
assert connection.execute("SELECT count(*) FROM strict_tractors WHERE NOT is_analysis_candidate OR NOT is_structurally_valid OR C_SEV NOT IN ('1','2') OR C_SEV IS NULL").fetchone()[0] == 0
assert connection.execute("SELECT count(*) FROM annual WHERE observed_tractor_collisions <> observed_fatal_tractor_collisions + observed_injury_tractor_collisions OR strict_tractor_collisions <> strict_fatal_tractor_collisions + strict_injury_tractor_collisions").fetchone()[0] == 0
assert validation['checks']['collision_key_unique'] is True
assert len(validation['annual_counts']) == connection.execute("SELECT count(*) FROM annual").fetchone()[0]
print(f"Independent checks passed: {all_keys[0]:,} collision rows.")
show_table("SELECT 'all collisions' AS population,count(*) AS collision_rows,min(C_YEAR) AS first_year,max(C_YEAR) AS last_year,count(DISTINCT C_YEAR) AS years FROM collisions UNION ALL SELECT 'observed tractor',count(*),min(C_YEAR),max(C_YEAR),count(DISTINCT C_YEAR) FROM tractors UNION ALL SELECT 'strict tractor sensitivity subset',count(*),min(C_YEAR),max(C_YEAR),count(DISTINCT C_YEAR) FROM strict_tractors")

Independent checks passed: 2,950,931 collision rows.


population,collision rows,first year,last year,years
all collisions,"2,950,931",1999,2021,23
observed tractor,"80,632",1999,2021,23
strict tractor sensitivity subset,"80,621",1999,2021,23


## Published benchmark reconciliation, 2012–2021

Fatal and injury columns below count unique collisions with a recorded road tractor, **before strict candidate exclusions**. This observed population is compared with the publication. A difference is the observed microdata count minus the publication count. `exact_match` means agreement for both severity categories; `unresolved_difference` remains an open source or definition issue.

Strict candidate counts are displayed separately below because conflicts can exclude an observed tractor event from analysis even when the aggregate observed count matches the publication. An aggregate match does not certify each record's vehicle classification. Never adjust individual records merely to force a published total.

In [4]:
show_table("""SELECT year, observed_fatal_tractor_collisions AS observed_fatal, published_fatal,
    fatal_difference, observed_injury_tractor_collisions AS observed_injury, published_injury,
    injury_difference, external_reconciliation
    FROM reconciliation WHERE year BETWEEN 2012 AND 2021 ORDER BY year""")
mismatches = connection.execute("SELECT year FROM reconciliation WHERE external_reconciliation='unresolved_difference' ORDER BY year").fetchall()
assert [row[0] for row in mismatches] == validation['checks']['external_mismatch_years']
display(Markdown("**Years with unresolved published differences:** " + (", ".join(str(row[0]) for row in mismatches) or "none in the available benchmark")))

year,observed fatal,published fatal,fatal difference,observed injury,published injury,injury difference,external reconciliation
2012,197,197,0,"3,103","3,103",0,exact_match
2013,195,195,0,"3,148","3,150",-2,unresolved_difference
2014,211,211,0,"3,216","3,216",0,exact_match
2015,194,194,0,"2,880","2,881",-1,unresolved_difference
2016,193,193,0,"2,816","2,817",-1,unresolved_difference
2017,214,216,-2,"2,962","2,962",0,unresolved_difference
2018,190,190,0,"2,800","2,800",0,exact_match
2019,192,196,-4,"2,679","2,581",98,unresolved_difference
2020,158,158,0,"1,964","1,964",0,exact_match
2021,169,169,0,"2,107","2,107",0,exact_match


**Years with unresolved published differences:** 2013, 2015, 2016, 2017, 2019

## Latest five years and quality flags

The primary count preserves every observed tractor event. The strict sensitivity count excludes collision-field or tractor-type conflicts under the build rules; these exclusions are not corrections to the primary population. Other warning flags remain visible and can constrain later factor or victim analyses. Vehicle-count disagreement and unknown vehicle types should not be silently imputed.

In [5]:
show_table("""SELECT year,all_casualty_collisions,observed_tractor_collisions,
    observed_fatal_tractor_collisions,observed_injury_tractor_collisions,
    strict_tractor_collisions,excluded_tractor_events
    FROM annual ORDER BY year DESC LIMIT 5""")

year,all casualty collisions,observed tractor collisions,observed fatal tractor collisions,observed injury tractor collisions,strict tractor collisions,excluded tractor events
2021,"83,590","2,276",169,"2,107","2,275",1
2020,"79,990","2,122",158,"1,964","2,116",6
2019,"105,782","2,871",192,"2,679","2,871",0
2018,"111,334","2,990",190,"2,800","2,990",0
2017,"114,408","3,176",214,"2,962","3,174",2


In [6]:
show_table("""SELECT year,duplicate_record_rows_removed,collision_field_conflicts,
    vehicle_type_conflicts,person_attribute_conflicts,vehicle_count_disagreements,
    nonnumeric_reported_vehicle_counts,unidentified_tractor_events,fatality_consistency_warnings
    FROM quality WHERE is_road_tractor ORDER BY year DESC LIMIT 5""")

year,duplicate record rows removed,collision field conflicts,vehicle type conflicts,person attribute conflicts,vehicle count disagreements,nonnumeric reported vehicle counts,unidentified tractor events,fatality consistency warnings
2021,0,0,2,2,7,0,0,0
2020,4,1,7,7,8,2,0,0
2019,1,0,1,1,7,0,0,0
2018,1,0,0,0,8,0,0,0
2017,0,0,3,3,6,0,0,0


### Special vehicle codes

These are frequencies of distinct source rows carrying special or nonnumeric vehicle codes, not collision or vehicle totals. The bounded preview focuses on the latest five years. The complete audit is saved in `special_vehicle_codes.csv`.

In [7]:
show_table("""SELECT year,V_TYPE,V_ID,person_record_rows
    FROM special WHERE year >= (SELECT max(year)-4 FROM annual)
    QUALIFY row_number() OVER (PARTITION BY year ORDER BY person_record_rows DESC,V_TYPE,V_ID) <= 3
    ORDER BY year DESC,person_record_rows DESC LIMIT 15""")

year,V TYPE,V ID,person record rows
2021,NN,99,"9,285"
2021,UU,01,"1,096"
2021,QQ,01,446
2020,NN,99,"9,624"
2020,UU,01,801
2020,QQ,01,397
2019,NN,99,"12,280"
2019,UU,01,761
2019,QQ,01,470
2018,NN,99,"12,940"


## Interpretation and next use

Use the saved status and reconciliation table together. Structural validation establishes a reproducible collision table with documented flags and a strict sensitivity subset; an external mismatch remains a limitation on claims of reproducing the published national series. A match in observed counts does not remove conflicts or exclusions in the stricter analysis candidates. Unknown codes, incomplete vehicle identifiers, and person-record warnings remain available for targeted follow-up.

The public casualty population cannot estimate all crashes, fault, or exposure-adjusted risk on its own. The next analytical step is a collision frequency and severity study with explicit source versions and exclusions. Validate person linkage before calculating deaths or injuries, and obtain a compatible exposure series before calculating risk per kilometre.

In [8]:
connection.close()
print("Validation companion completed. The source build outputs were read without modification.")

Validation companion completed. The source build outputs were read without modification.
